# 03 — Demo

**PPE Compliance Agent — ITAI 1378 Final Project**

Interactive Gradio demo that runs the full agent pipeline (all 6 stages) on an uploaded photo
and shows the annotated result plus the agent's reasoning trail — not just a label.

In [ ]:
!pip install -q gradio

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import cv2
import gradio as gr
from tools.detector import PPEDetector
from tools.preprocessing import validate_and_load
from agents.reasoning import decide_compliance

In [ ]:
detector = PPEDetector(weights_path="../models/trained/best_yolov8n_ppe.pt")
if detector.using_fallback:
    print("NOTE: running with fallback generic weights — train the model in 01_exploration.ipynb ",
          "or place best_yolov8n_ppe.pt in models/trained/ for real mask detection.")

In [ ]:
def run_agent_on_image(image):
    """Runs stages 2-4 live for the Gradio demo (stage 1 is the upload itself)."""
    # Gradio gives us a numpy array directly; write to a temp file so our
    # preprocessing stage exercises the exact same validation path as the CLI.
    tmp_path = "/tmp/gradio_upload.jpg"
    cv2.imwrite(tmp_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

    pre = validate_and_load(tmp_path)
    if not pre.valid:
        return image, f"❌ Input rejected: {pre.reason}"

    detections, raw_result = detector.detect(pre.image_array)
    decision = decide_compliance(detections)

    annotated = raw_result.plot()[..., ::-1]  # BGR -> RGB

    status_emoji = {"COMPLIANT": "✅", "NON_COMPLIANT": "⚠️", "NO_DETECTION": "❓"}
    report = (
        f"{status_emoji.get(decision.status, '')} **{decision.status}**\n\n"
        f"**Rule fired:** {decision.rule_fired}\n\n"
        f"**Reasoning:** {decision.explanation}"
    )
    return annotated, report

In [ ]:
demo = gr.Interface(
    fn=run_agent_on_image,
    inputs=gr.Image(type="numpy", label="Upload caregiver photo"),
    outputs=[gr.Image(label="Detection Result"), gr.Markdown(label="Agent Decision")],
    title="PPE Compliance Agent",
    description="Upload a photo to see the full perceive-reason-act pipeline in action. ITAI 1378 final project.",
    examples=[
        "../data/sample/caregiver_car_nomask.jpg",
        "../data/sample/caregiver_home_nomask.jpg",
        "../data/sample/caregiver_clinic_masked.jpg",
    ],
)

demo.launch(share=True)  # public link — use for screen-recording the demo video